# On-Device Deployment: Quantization & Cross-Platform Profiling
This notebook quantizes the trained ResNet-18 ECG model from FP32 to INT8 and measures
the impact on diagnostic accuracy (AUC). It then profiles deployment feasibility across
multiple platforms and device tiers.

### Pipeline:
1. Load ONNX model exported from training notebook.
2. Apply post-training static quantization via ONNX Runtime (INT8).
3. Evaluate FP32 vs INT8 on Shaoxing test set.
4. **Qualcomm AI Hub**: Quantize, compile, and profile on Snapdragon devices (flagship → ultra-low-cost).
5. **ExecuTorch**: PyTorch-native mobile export (vendor-neutral).
6. **ONNX Runtime**: Host-side latency benchmarking (vendor-neutral).
7. **Apple Core ML**: Convert and profile on Apple Silicon (Neural Engine).

In [1]:
import numpy as np
import os
import pandas as pd
import wfdb
from sklearn.metrics import roc_auc_score
import onnxruntime as ort
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantType, QuantFormat

# Helper: pad/truncate signals to fixed length for static ONNX input
def pad_or_truncate(sig, length=1000):
    if len(sig) >= length:
        return sig[:length]
    return np.pad(sig, (0, length - len(sig)))



## 1. Load and Verify FP32 ONNX Model

In [2]:
onnx_fp32_path = "resnet18_ecg.onnx"

# ONNX models exported by PyTorch may store weights in a separate .data file.
# Convert to a single self-contained file for compatibility with AI Hub and other tools.
import onnx
from onnx.external_data_helper import convert_model_from_external_data

onnx_model = onnx.load(onnx_fp32_path, load_external_data=True)
convert_model_from_external_data(onnx_model)
onnx_fp32_inline_path = "resnet18_ecg_inline.onnx"
onnx.save(onnx_model, onnx_fp32_inline_path)

# Use the inline version from here on
onnx_fp32_path = onnx_fp32_inline_path

session_fp32 = ort.InferenceSession(onnx_fp32_path)
input_name = session_fp32.get_inputs()[0].name
output_name = session_fp32.get_outputs()[0].name

# Test with dummy input
dummy = np.random.randn(1, 1, 1000).astype(np.float32)
result = session_fp32.run([output_name], {input_name: dummy})
print(f"FP32 model loaded. Test output: {result[0]}")
fp32_size = os.path.getsize(onnx_fp32_path) / (1024*1024)
print(f"FP32 model size: {fp32_size:.2f} MB")


FP32 model loaded. Test output: [[-4.8573294]]
FP32 model size: 14.73 MB


## 2. Prepare Shaoxing Evaluation Data

In [3]:
shaoxing_path = '../data/a-large-scale-12-lead-electrocardiogram-database-for-arrhythmia-study-1.0.0/'
MI_CODE = '164865005'

def load_shaoxing_samples(base_path, lead_idx=0, limit=5000):
    """Load Shaoxing ECG signals and labels as numpy arrays."""
    records_file = os.path.join(base_path, 'RECORDS')
    with open(records_file, 'r') as f:
        subfolders = [line.strip() for line in f if line.strip()]
    
    signals, labels = [], []
    for sub in subfolders:
        if len(signals) >= limit:
            break
        sub_path = os.path.join(base_path, sub)
        if not os.path.exists(sub_path):
            continue
        files = [f.replace('.hea', '') for f in os.listdir(sub_path) if f.endswith('.hea')]
        for fname in files:
            if len(signals) >= limit:
                break
            try:
                rec_path = os.path.join(base_path, sub, fname)
                signal, meta = wfdb.rdsamp(rec_path)
                lead_signal = signal[::5, lead_idx]  # downsample 500Hz -> 100Hz
                lead_signal = (lead_signal - np.mean(lead_signal)) / (np.std(lead_signal) + 1e-8)
                
                dx_list = []
                for comment in meta['comments']:
                    if 'Dx:' in comment:
                        dx_list = [c.strip() for c in comment.split('Dx:')[1].strip().split(',')]
                        break
                
                signals.append(lead_signal.astype(np.float32))
                labels.append(1 if MI_CODE in dx_list else 0)
            except Exception:
                continue
    
    return signals, labels

signals, labels = load_shaoxing_samples(shaoxing_path, limit=5000)
print(f"Loaded {len(signals)} Shaoxing samples | MI prevalence: {np.mean(labels)*100:.2f}%")

Loaded 5000 Shaoxing samples | MI prevalence: 0.24%


## 3. Quantize to INT8

In [ ]:
onnx_int8_path = "resnet18_ecg_int8.onnx"

if os.path.exists(onnx_int8_path):
    print(f"ONNX Runtime INT8 model already exists, skipping quantization.")
else:
    from onnxruntime.quantization import preprocess
    
    # Pre-process the model (graph optimization) to eliminate quantization warnings
    onnx_preprocessed_path = "resnet18_ecg_preprocessed.onnx"
    preprocess.quant_pre_process(onnx_fp32_path, onnx_preprocessed_path)
    print("ONNX model pre-processed for quantization.")
    
    class ECGCalibrationReader(CalibrationDataReader):
        """Feeds a subset of real ECG signals for calibration during static quantization."""
        def __init__(self, signals, input_name, num_samples=200):
            self.signals = [pad_or_truncate(sig) for sig in signals[:num_samples]]
            self.input_name = input_name
            self.idx = 0
        
        def get_next(self):
            if self.idx >= len(self.signals):
                return None
            sig = self.signals[self.idx]
            self.idx += 1
            return {self.input_name: sig.reshape(1, 1, 1000)}

    calibration_reader = ECGCalibrationReader(signals, input_name, num_samples=200)

    quantize_static(
        model_input=onnx_preprocessed_path,
        model_output=onnx_int8_path,
        calibration_data_reader=calibration_reader,
        quant_format=QuantFormat.QDQ,
        weight_type=QuantType.QInt8,
        activation_type=QuantType.QInt8,
    )
    print("ONNX Runtime INT8 quantization complete.")

fp32_size = os.path.getsize(onnx_fp32_path) / (1024 * 1024)
int8_size = os.path.getsize(onnx_int8_path) / (1024 * 1024)
print(f"FP32 model size: {fp32_size:.2f} MB")
print(f"INT8 model size: {int8_size:.2f} MB")
print(f"Compression ratio: {fp32_size / int8_size:.1f}x")


## 4. Evaluate FP32 vs INT8 on Shaoxing

In [5]:
def evaluate_onnx(session, signals, labels):
    """Run inference on all signals and compute AUC."""
    input_name = session.get_inputs()[0].name
    output_name = session.get_outputs()[0].name
    
    preds = []
    for sig in signals:
        inp = pad_or_truncate(sig).reshape(1, 1, 1000)
        logit = session.run([output_name], {input_name: inp})[0][0, 0]
        prob = 1 / (1 + np.exp(-logit))  # sigmoid
        preds.append(prob)
    
    return roc_auc_score(labels, preds)

# session_fp32 already loaded in cell 3
session_int8 = ort.InferenceSession(onnx_int8_path)

auc_fp32 = evaluate_onnx(session_fp32, signals, labels)
auc_int8 = evaluate_onnx(session_int8, signals, labels)

print(f"FP32 AUC: {auc_fp32:.4f}")
print(f"INT8 AUC: {auc_int8:.4f}")
print(f"AUC delta: {auc_int8 - auc_fp32:+.4f}")


FP32 AUC: 0.6304
INT8 AUC: 0.6328
AUC delta: +0.0024


## 5. ONNX Runtime Summary

Full comparison table (all platforms) is in the final cell.


The full comparison table (ONNX Runtime + QAI Hub + Core ML) is at the end of this notebook.


---

## 6. Qualcomm AI Hub: Quantize, Compile & Profile

Below, we use [Qualcomm AI Hub](https://aihub.qualcomm.com) to quantize, compile, and profile
on real Snapdragon chipsets — giving us actual on-device latency and memory numbers.

**Prerequisites:**
- `pip install qai-hub`
- Configure your API token: `qai-hub configure --api_token YOUR_TOKEN`


In [6]:
import qai_hub as hub

client = hub.Client()

# Verify connection
devices_list = client.get_devices()
print(f"Connected to Qualcomm AI Hub. {len(devices_list)} devices available.")


Connected to Qualcomm AI Hub. 81 devices available.


### 6a. Quantize on AI Hub (INT8 weights + INT8 activations)

In [7]:
# Build calibration data from real Shaoxing ECG signals
calibration_data = {input_name: [pad_or_truncate(sig).reshape(1, 1, 1000) for sig in signals[:200]]}

quantize_job = client.submit_quantize_job(
    model=onnx_fp32_path,
    calibration_data=calibration_data,
    weights_dtype=hub.QuantizeDtype.INT8,
    activations_dtype=hub.QuantizeDtype.INT8,
    name="resnet18_ecg_int8_qai"
)

print(f"Quantization job submitted: {quantize_job}")
print("Waiting for completion...")
quantize_job.wait()

status = quantize_job.get_status()
print(f"Job status: {status.code}")
if "FAILED" in str(status.code):
    raise RuntimeError(f"AI Hub quantization failed: {status.message}")

# Download and extract the quantized model (AI Hub returns a zip with model.onnx + model.data)
import zipfile
import onnx
from onnx.external_data_helper import convert_model_from_external_data

downloaded_path = quantize_job.download_target_model("resnet18_ecg_qai_int8")
print(f"Downloaded: {downloaded_path}")

qai_model_dir = "resnet18_ecg_qai_int8_dir"
os.makedirs(qai_model_dir, exist_ok=True)

if downloaded_path.endswith(".zip"):
    with zipfile.ZipFile(downloaded_path, 'r') as zf:
        zf.extractall(qai_model_dir)
        print(f"Extracted to {qai_model_dir}/: {zf.namelist()}")

# Find model.onnx inside extracted directory
qai_onnx_path = None
for root, dirs, files in os.walk(qai_model_dir):
    if "model.onnx" in files:
        qai_onnx_path = os.path.join(root, "model.onnx")
        break

if qai_onnx_path is None:
    raise FileNotFoundError(f"model.onnx not found in {qai_model_dir}")

# Convert to inline weights for easy loading with ONNX Runtime
qai_model = onnx.load(qai_onnx_path, load_external_data=True)
convert_model_from_external_data(qai_model)
qai_quantized_path = "resnet18_ecg_qai_int8.onnx"
onnx.save(qai_model, qai_quantized_path)

qai_int8_size = os.path.getsize(qai_quantized_path) / (1024 * 1024)
print(f"QAI Hub INT8 model size: {qai_int8_size:.2f} MB")


Uploading resnet18_ecg_inline.onnx


100%|██████████| 14.7M/14.7M [00:04<00:00, 3.14MB/s]
Uploading dataset: 1.07MB [00:00, 1.44MB/s]                           779kB/s]  


Scheduled quantize job (jgklrjko5) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jgklrjko5/

Quantization job submitted: Job(job_id=jgklrjko5, model_id=mqp1g6ovq)
Waiting for completion...
Waiting for quantize job (jgklrjko5) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          
Job status: SUCCESS


resnet18_ecg_qai_int8.onnx.zip: 100%|██████████| 7.47M/7.47M [00:00<00:00, 18.3MB/s]

Downloaded model to resnet18_ecg_qai_int8.onnx.zip
Downloaded: resnet18_ecg_qai_int8.onnx.zip
Extracted to resnet18_ecg_qai_int8_dir/: ['job_jgklrjko5_qdq_onnx/', 'job_jgklrjko5_qdq_onnx/model.data', 'job_jgklrjko5_qdq_onnx/model.onnx']
QAI Hub INT8 model size: 14.77 MB


In [ ]:
# Profile across a range of devices: flagship to ultra-low-cost
# S24 gets the quantized model compiled to QNN
# Older devices get the FP32 model compiled to TFLite
# (QAI Hub's TFLite backend can't handle quantized 1D conv transpose ops)

# Upload the QAI quantized model for flagship, FP32 for budget devices
devices = {
    "flagship": {
        "device": hub.Device("Samsung Galaxy S24 (Family)"),
        "label": "Samsung Galaxy S24 (Snapdragon 8 Gen 3, ~$800)",
        "model": qai_quantized_path,  # INT8 quantized ONNX
        "runtime": "--target_runtime qnn_context_binary",
        "note": "INT8 via QNN",
    },
    "budget": {
        "device": hub.Device("Xiaomi Redmi Note 10 5G"),
        "label": "Xiaomi Redmi Note 10 5G (Snapdragon 678, ~$80-100 used)",
        "model": onnx_fp32_path,  # FP32 — TFLite can't compile quantized 1D conv
        "runtime": None,
        "note": "FP32 via TFLite",
    },
    "ultra_low": {
        "device": hub.Device("Samsung Galaxy Tab A8 (2021)"),
        "label": "Samsung Galaxy Tab A8 (Snapdragon 429, ~$100)",
        "model": onnx_fp32_path,  # FP32 — TFLite can't compile quantized 1D conv
        "runtime": None,
        "note": "FP32 via TFLite",
    },
}

compile_jobs = {}
for key, cfg in devices.items():
    kwargs = dict(
        model=cfg["model"],
        device=cfg["device"],
        name=f"resnet18_ecg_compile_{key}",
    )
    if cfg["runtime"]:
        kwargs["options"] = cfg["runtime"]
    compile_jobs[key] = client.submit_compile_job(**kwargs)
    print(f"Compile job submitted: {cfg['label']} ({cfg['note']})")

print("\nWaiting for all compilations...")
failed = []
for key, job in compile_jobs.items():
    job.wait()
    status = job.get_status()
    if "FAILED" in str(status.code):
        print(f"  {key}: FAILED — {status.message[:120]}")
        failed.append(key)
    else:
        print(f"  {key}: done")

for key in failed:
    del compile_jobs[key]
    del devices[key]

if compile_jobs:
    print(f"\nCompilation complete for {len(compile_jobs)} devices.")
else:
    raise RuntimeError("All compile jobs failed.")




In [9]:
# Profile all successfully compiled devices
profile_jobs = {}
for key, cfg in devices.items():
    profile_jobs[key] = client.submit_profile_job(
        model=compile_jobs[key].get_target_model(),
        device=cfg["device"],
        name=f"resnet18_ecg_profile_{key}",
    )
    print(f"Profile job submitted: {cfg['label']}")

print("\nWaiting for all profiles...")
for key, job in profile_jobs.items():
    job.wait()
    status = job.get_status()
    if "FAILED" in str(status.code):
        print(f"  {key}: FAILED — {status.message}")
    else:
        print(f"  {key}: done")

print()
for key, cfg in devices.items():
    if key not in profile_jobs:
        continue
    status = profile_jobs[key].get_status()
    if "FAILED" in str(status.code):
        continue
    profile_data = profile_jobs[key].download_profile()
    print("=" * 65)
    print(f"  {cfg['label']}")
    print("=" * 65)
    # Profile data may be a dict or printable object — handle both
    if isinstance(profile_data, dict):
        summary = profile_data.get("execution_summary", profile_data)
        for k, v in summary.items():
            print(f"  {k}: {v}")
    else:
        print(profile_data)
    print()


Scheduled profile job (jgzxv16k5) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jgzxv16k5/

Profile job submitted: Samsung Galaxy S24 (Snapdragon 8 Gen 3, ~$800)
Scheduled profile job (j5wdmjj6g) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/j5wdmjj6g/

Profile job submitted: Xiaomi Redmi Note 10 5G (Snapdragon 678, ~$80-100 used)
Scheduled profile job (jg93966lg) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jg93966lg/

Profile job submitted: Samsung Galaxy Tab A8 (Snapdragon 429, ~$100)

Waiting for all profiles...
Waiting for profile job (jgzxv16k5) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          
  flagship: done
Waiting for profile job (j5wdmjj6g) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          
  budget: done
Waiting for profile job (jg93966lg) completion. Ty

### 6d. Evaluate QAI Hub Quantized Model on Shaoxing

In [10]:
# Evaluate the QAI Hub quantized ONNX locally (same method as ONNX Runtime INT8 above)
session_qai = ort.InferenceSession(qai_quantized_path)
auc_qai = evaluate_onnx(session_qai, signals, labels)

print(f"FP32 AUC:               {auc_fp32:.4f}")
print(f"ONNX Runtime INT8 AUC:  {auc_int8:.4f}")
print(f"QAI Hub INT8 AUC:       {auc_qai:.4f}")
print(f"\nQAI Hub AUC delta:      {auc_qai - auc_fp32:+.4f}")

FP32 AUC:               0.6304
ONNX Runtime INT8 AUC:  0.6328
QAI Hub INT8 AUC:       0.6408

QAI Hub AUC delta:      +0.0104


In [ ]:
summary_full = pd.DataFrame({
    'Model': ['FP32 (Original)', 'INT8 (ONNX Runtime)', 'INT8 (Qualcomm AI Hub)'],
    'Size (MB)': [f'{fp32_size:.2f}', f'{int8_size:.2f}', f'{qai_int8_size:.2f}'],
    'Shaoxing AUC': [f'{auc_fp32:.4f}', f'{auc_int8:.4f}', f'{auc_qai:.4f}'],
    'AUC Retention': [
        'baseline',
        f'{(auc_int8/auc_fp32)*100:.1f}%',
        f'{(auc_qai/auc_fp32)*100:.1f}%'
    ]
})

print("=== Quantization Impact ===")
print(summary_full.to_string(index=False))
print()
print("Note: QAI Hub INT8 model uses QDQ format (quantize-dequantize nodes alongside")
print("full-precision graph structure), so file size is similar to FP32. The ONNX Runtime")
print("INT8 model achieves the real 3.9x compression — this is the number to cite.")
print()
print("=== Qualcomm AI Hub On-Device Profiling (INT8) ===")
for key, cfg in devices.items():
    print(f"  {cfg['label']}: see profile output above")
print()
print("All profiles use the Qualcomm AI Hub INT8 quantized model.")
print("If inference on the ultra-low-cost device is under ~100ms,")
print("real-time ECG screening is feasible on hardware affordable in LMICs.")


---

## 7. ExecuTorch: PyTorch-Native Mobile Runtime

ExecuTorch is PyTorch's equivalent of TensorFlow Lite — it's the official path from
`torch.nn.Module` to on-device inference. Since our model is already in PyTorch, this
avoids any cross-framework conversion (no ONNX→TF→TFLite chain needed).

ExecuTorch runs on any ARM device regardless of chipset vendor.

**Prerequisites:** `pip install executorch`

In [12]:
import torch
import glob
from torch.export import export
from executorch.exir import to_edge_transform_and_lower

# --- Shared model architecture (used by ExecuTorch + Core ML) ---
def conv3x1(in_planes, out_planes, stride=1):
    return torch.nn.Conv1d(in_planes, out_planes, kernel_size=3, stride=stride, padding=1, bias=False)

class BasicBlock1D(torch.nn.Module):
    expansion = 1
    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super().__init__()
        self.conv1 = conv3x1(inplanes, planes, stride)
        self.bn1 = torch.nn.BatchNorm1d(planes)
        self.relu = torch.nn.ReLU(inplace=True)
        self.conv2 = conv3x1(planes, planes)
        self.bn2 = torch.nn.BatchNorm1d(planes)
        self.downsample = downsample

    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            residual = self.downsample(x)
        out += residual
        return self.relu(out)

class ResNet1D(torch.nn.Module):
    def __init__(self, block, layers, num_classes=1):
        self.inplanes = 64
        super().__init__()
        self.conv1 = torch.nn.Conv1d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = torch.nn.BatchNorm1d(64)
        self.relu = torch.nn.ReLU(inplace=True)
        self.maxpool = torch.nn.MaxPool1d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)
        self.avgpool = torch.nn.AdaptiveAvgPool1d(1)
        self.fc = torch.nn.Linear(512, num_classes)

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = torch.nn.Sequential(
                torch.nn.Conv1d(self.inplanes, planes * block.expansion, kernel_size=1, stride=stride, bias=False),
                torch.nn.BatchNorm1d(planes * block.expansion),
            )
        layers = [block(self.inplanes, planes, stride, downsample)]
        self.inplanes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.inplanes, planes))
        return torch.nn.Sequential(*layers)

    def forward(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        x = self.avgpool(x)
        return self.fc(x.view(x.size(0), -1))

# --- Load checkpoint once (reused by ExecuTorch + Core ML) ---
ckpt_files = sorted(glob.glob("../checkpoints/resnet18_finetuned_*.pt"))
if not ckpt_files:
    raise FileNotFoundError("No fine-tuned checkpoint found. Run the training notebook first.")
ckpt_path = ckpt_files[-1]
print(f"Loading checkpoint: {ckpt_path}")

pt_model = ResNet1D(BasicBlock1D, [2, 2, 2, 2])
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
pt_model.load_state_dict(ckpt['model_state_dict'])
pt_model.eval()

# --- ExecuTorch export ---
example_input = (torch.randn(1, 1, 1000),)
exported = export(pt_model, example_input)
et_program = to_edge_transform_and_lower(exported).to_executorch()

et_path = "resnet18_ecg.pte"
with open(et_path, "wb") as f:
    f.write(et_program.buffer)

et_size = os.path.getsize(et_path) / (1024 * 1024)
print(f"ExecuTorch model exported to: {et_path}")
print(f"ExecuTorch model size: {et_size:.2f} MB")


W0402 19:15:26.871000 5334 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading checkpoint: checkpoints/resnet18_finetuned_20260402_173412.pt


/Users/gabrielbutler/.pyenv/versions/3.11.9/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/Users/gabrielbutler/.pyenv/versions/3.11.9/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


ExecuTorch model exported to: resnet18_ecg.pte
ExecuTorch model size: 14.73 MB


---

## 8. ONNX Runtime: Host-Side Latency Benchmark

Vendor-neutral latency measurement using ONNX Runtime on the local CPU.
This provides a baseline that doesn't depend on any specific mobile chipset.

In [13]:
import time

def benchmark_onnx(session, signals, n=100):
    """Benchmark ONNX Runtime inference latency."""
    input_name = session.get_inputs()[0].name
    output_name = session.get_outputs()[0].name
    
    # Warm-up
    for sig in signals[:10]:
        session.run([output_name], {input_name: pad_or_truncate(sig).reshape(1, 1, 1000)})
    
    latencies = []
    for sig in signals[:n]:
        inp = pad_or_truncate(sig).reshape(1, 1, 1000)
        start = time.perf_counter()
        session.run([output_name], {input_name: inp})
        latencies.append((time.perf_counter() - start) * 1000)
    
    return latencies

# Reuse session_fp32 (cell 3) and session_int8 (cell 9)
lat_fp32 = benchmark_onnx(session_fp32, signals)
lat_int8 = benchmark_onnx(session_int8, signals)

print(f"ONNX Runtime host-side latency (n=100):")
print(f"  FP32:  mean {np.mean(lat_fp32):.2f} ms | median {np.median(lat_fp32):.2f} ms | p95 {np.percentile(lat_fp32, 95):.2f} ms")
print(f"  INT8:  mean {np.mean(lat_int8):.2f} ms | median {np.median(lat_int8):.2f} ms | p95 {np.percentile(lat_int8, 95):.2f} ms")
print(f"  Speedup: {np.mean(lat_fp32) / np.mean(lat_int8):.2f}x")


ONNX Runtime host-side latency (n=100):
  FP32:  mean 0.75 ms | median 0.74 ms | p95 0.84 ms
  INT8:  mean 0.27 ms | median 0.27 ms | p95 0.29 ms
  Speedup: 2.78x


---

## 9. Apple Core ML: Neural Engine Profiling

Convert the model to Core ML format and benchmark on this Mac's Apple Silicon.
Core ML can dispatch to the Neural Engine, GPU, or CPU — the framework chooses automatically.
This demonstrates cross-platform feasibility beyond Android/Snapdragon.

**Prerequisites:** `pip install coremltools`

In [14]:
import coremltools as ct

coreml_fp32_path = "resnet18_ecg.mlpackage"
coreml_int8_path = "resnet18_ecg_int8.mlpackage"

if os.path.exists(coreml_fp32_path) and os.path.exists(coreml_int8_path):
    print("Core ML models already exist, loading from disk.")
    mlmodel_fp32 = ct.models.MLModel(coreml_fp32_path)
    mlmodel_int8 = ct.models.MLModel(coreml_int8_path)
else:
    # Export and decompose to ATEN dialect (required by coremltools)
    exported_program = torch.export.export(pt_model, (torch.randn(1, 1, 1000),))
    exported_program = exported_program.run_decompositions({})

    mlmodel_fp32 = ct.convert(
        exported_program,
        minimum_deployment_target=ct.target.macOS13,
    )
    mlmodel_fp32.save(coreml_fp32_path)
    print(f"Core ML FP32 model saved to {coreml_fp32_path}")

    # Quantize to INT8 within Core ML
    from coremltools.optimize.coreml import (
        OpLinearQuantizerConfig,
        OptimizationConfig,
        linear_quantize_weights,
    )

    quant_config = OptimizationConfig(
        global_config=OpLinearQuantizerConfig(mode="linear_symmetric", dtype="int8")
    )
    mlmodel_int8 = linear_quantize_weights(mlmodel_fp32, config=quant_config)
    mlmodel_int8.save(coreml_int8_path)
    print(f"Core ML INT8 model saved to {coreml_int8_path}")


Core ML models already exist, loading from disk.


In [15]:
# Benchmark Core ML inference on Apple Silicon
# predict() uses the Neural Engine / GPU / CPU as the system decides

# Get the actual input name from the model (torch.export may assign 'x' instead of 'ecg_lead')
coreml_input_name = mlmodel_fp32.input_description._fd_spec[0].name
print(f"Core ML input name: '{coreml_input_name}'")

def benchmark_coreml(model, signals, input_name, n=100):
    """Benchmark Core ML inference latency."""
    # Warm-up
    for sig in signals[:10]:
        inp = {input_name: pad_or_truncate(sig).reshape(1, 1, 1000)}
        model.predict(inp)
    
    latencies = []
    for sig in signals[:n]:
        inp = {input_name: pad_or_truncate(sig).reshape(1, 1, 1000)}
        start = time.perf_counter()
        model.predict(inp)
        latencies.append((time.perf_counter() - start) * 1000)
    return latencies

lat_coreml_fp32 = benchmark_coreml(mlmodel_fp32, signals, coreml_input_name)
lat_coreml_int8 = benchmark_coreml(mlmodel_int8, signals, coreml_input_name)

print(f"Core ML latency on Apple Silicon (n=100):")
print(f"  FP32:  mean {np.mean(lat_coreml_fp32):.2f} ms | median {np.median(lat_coreml_fp32):.2f} ms | p95 {np.percentile(lat_coreml_fp32, 95):.2f} ms")
print(f"  INT8:  mean {np.mean(lat_coreml_int8):.2f} ms | median {np.median(lat_coreml_int8):.2f} ms | p95 {np.percentile(lat_coreml_int8, 95):.2f} ms")
print(f"  Speedup: {np.mean(lat_coreml_fp32) / np.mean(lat_coreml_int8):.2f}x")


Core ML input name: 'x'
Core ML latency on Apple Silicon (n=100):
  FP32:  mean 0.18 ms | median 0.17 ms | p95 0.21 ms
  INT8:  mean 0.17 ms | median 0.16 ms | p95 0.18 ms
  Speedup: 1.07x


In [16]:
# Evaluate Core ML models on Shaoxing for AUC comparison
def evaluate_coreml(model, signals, labels, input_name):
    preds = []
    for sig in signals:
        inp = {input_name: pad_or_truncate(sig).reshape(1, 1, 1000)}
        out = model.predict(inp)
        logit = list(out.values())[0].flatten()[0]
        prob = 1 / (1 + np.exp(-logit))
        preds.append(prob)
    return roc_auc_score(labels, preds)

auc_coreml_fp32 = evaluate_coreml(mlmodel_fp32, signals, labels, coreml_input_name)
auc_coreml_int8 = evaluate_coreml(mlmodel_int8, signals, labels, coreml_input_name)

print(f"Core ML FP32 AUC: {auc_coreml_fp32:.4f}")
print(f"Core ML INT8 AUC: {auc_coreml_int8:.4f}")
print(f"AUC delta:        {auc_coreml_int8 - auc_coreml_fp32:+.4f}")


Core ML FP32 AUC: 0.6302
Core ML INT8 AUC: 0.6301
AUC delta:        -0.0001
